In [5]:
import time
import random

NOF_PACKETS = 5


def custom_rand(a: int) -> int:
    """Replicates the pseudo-random generator logic from the C code."""
    n = (random.randint(0, 9)) % a
    return 1 if n == 0 else n

def main():
    p_sz_rm = 0  # Tracks bytes currently remaining in the bucket

    # 1. Generate random sizes for the incoming packets (multiples of 10)
    packet_sz = [custom_rand(6) * 10 for _ in range(NOF_PACKETS)]

    # 2. Display the sizes of the generated incoming packets
    print("--- Generated Incoming Packets ---")
    for i, size in enumerate(packet_sz):
        print(f"packet[{i}]: {size} bytes")
    print("-----------------------------------\n")

    # 3. Get bucket specifications from the user
    try:
        o_rate = int(input("Enter the Output rate (bytes/sec): "))
        b_size = int(input("Enter the Bucket Size (bytes): "))
    except ValueError:
        print("Invalid input. Please enter integers only.")
        return

    # 4. Process each packet sequentially through the Leaky Bucket logic
    for i in range(NOF_PACKETS):
        current_packet = packet_sz[i]

        # Check if adding the incoming packet causes a bucket overflow
        if (current_packet + p_sz_rm) > b_size:
            if current_packet > b_size:
                # Case A: Single packet is inherently larger than the whole bucket
                print(f"\n\n[REJECTED] Incoming packet size ({current_packet} bytes) is Greater than bucket capacity ({b_size} bytes)")
            else:
                # Case B: Packet fits the capacity, but the bucket is already too full
                print("\n\n[REJECTED] Bucket capacity exceeded - PACKETS REJECTED!!")
        else:
            # Packet is successfully admitted into the bucket buffer
            p_sz_rm += current_packet
            print(f"\n\n[ADMITTED] Incoming Packet size: {current_packet} bytes")
            print(f"Bytes remaining in bucket to Transmit: {p_sz_rm}")

            # Determine a simulated transmission duration (processing clock cycles)
            p_time = custom_rand(4) * 10
            print(f"Time allocated for transmission window: {p_time} units")

            # Simulate the clock cycles leaking data at the specified output rate
            for clk in range(10, p_time + 1, 10):
                time.sleep(1) # Delays execution by 1 second to simulate active transmission ticks

                if p_sz_rm > 0:
                    # If data remains, leak/transmit up to the steady output rate
                    if p_sz_rm <= o_rate:
                        op = p_sz_rm
                        p_sz_rm = 0
                    else:
                        op = o_rate
                        p_sz_rm -= o_rate
                    print(f"\n  -> Packet segment of size {op} Transmitted")
                    print(f"     ---- Bytes Remaining in Bucket: {p_sz_rm}")
                else:
                    # If the bucket becomes completely empty before the clock window expires
                    print(f"\n  -> Time left in transmission window: {p_time - clk} units")
                    print("     No packets remaining to transmit!")

    print("\n\nProcessing complete.")

if __name__ == "__main__":
    main()


--- Generated Incoming Packets ---
packet[0]: 50 bytes
packet[1]: 10 bytes
packet[2]: 40 bytes
packet[3]: 40 bytes
packet[4]: 40 bytes
-----------------------------------

Enter the Output rate (bytes/sec): 5
Enter the Bucket Size (bytes): 100


[ADMITTED] Incoming Packet size: 50 bytes
Bytes remaining in bucket to Transmit: 50
Time allocated for transmission window: 20 units

  -> Packet segment of size 5 Transmitted
     ---- Bytes Remaining in Bucket: 45

  -> Packet segment of size 5 Transmitted
     ---- Bytes Remaining in Bucket: 40


[ADMITTED] Incoming Packet size: 10 bytes
Bytes remaining in bucket to Transmit: 50
Time allocated for transmission window: 30 units

  -> Packet segment of size 5 Transmitted
     ---- Bytes Remaining in Bucket: 45

  -> Packet segment of size 5 Transmitted
     ---- Bytes Remaining in Bucket: 40

  -> Packet segment of size 5 Transmitted
     ---- Bytes Remaining in Bucket: 35


[ADMITTED] Incoming Packet size: 40 bytes
Bytes remaining in bucket t